**This code is to use shape descriptors for classification using multi-layer perceptrons (MLPs):**

In [1]:
# Model training hyperparameters
MODEL = "pasc_ann"
DEVICE_ID = 1
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3
OPTIMIZER = "AdamW"
LOG_DIR = f"./{MODEL}_logs/"
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# --- Replace NaNs within each species group ---
def fill_by_class_mean(df, class_col="species"):
    df = df.replace(0, np.nan)
    df = df.dropna(axis=1, how='all')
    df_numeric = df.select_dtypes(include=[np.number])
    # Fill NaNs in numeric columns using the class-wise mean
    df[df_numeric.columns] = df.groupby(class_col)[df_numeric.columns].transform(
        lambda x: x.fillna(x.mean())
    )
    # Step 2: fill any remaining NaNs globally (for columns that were all NaN in a class)
    df[df_numeric.columns] = df[df_numeric.columns].fillna(df[df_numeric.columns].mean())
    return df

In [4]:
train_species_df = pd.read_csv(os.path.join(DATA_DIR, 'train_aug_labels.csv'))
val_species_df = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))
test_species_df = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))
train_species_df.shape, val_species_df.shape, test_species_df.shape

((34722, 2), (1158, 2), (771, 2))

In [5]:
train_feat_path = r"./predicted_shape_features/beemachine_partwhole_v5_train.csv"
val_feat_path   = r"./predicted_shape_features/beemachine_partwhole_v5_val.csv"
test_feat_path  = r"./predicted_shape_features/beemachine_partwhole_v5_test.csv"
train_feat_df = pd.read_csv(train_feat_path) 
val_feat_df = pd.read_csv(val_feat_path) 
test_feat_df = pd.read_csv(test_feat_path)
train_feat_df.shape, val_feat_df.shape, test_feat_df.shape

((34722, 938), (1158, 938), (771, 938))

In [6]:
train_df = pd.merge(train_feat_df, train_species_df, on='image')
val_df = pd.merge(val_feat_df, val_species_df, on='image')
test_df = pd.merge(test_feat_df, test_species_df, on='image')
train_df.shape, val_df.shape, test_df.shape

((34722, 939), (1158, 939), (771, 939))

In [7]:
train_df.head(2)

,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,Bombus_distinguendus_GBIF_iNat_2573822774_2-jp...,3925.0,368.46910,1.107346,0.588015,0.772942,0.429514,-1.047061,0.363285,0.096940,...,6.315095,58.42423,30.060743,43.24497,0.175741,2.572449,0.112332,0.639192,0.248476,Bombus_distinguendus
1,Bombus_bifarius_47362646_1_1_jpg.rf.bd7d10af8d...,865.0,129.49748,2.422290,0.706699,0.929109,0.910807,1.360611,0.648192,0.587168,...,6.093960,94.43765,26.321207,87.51460,3.728448,0.028823,0.094577,0.025366,0.880057,Bombus_bifarius


In [8]:
val_df.head(2)

,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,BBW_Bombus_appositus_23855_e5f978d9-d6e7-4322-...,2691.0,266.59293,1.182834,0.542540,0.749373,0.534091,-0.748358,0.475802,0.154573,...,5.619246,69.066900,27.150719,57.412890,0.121364,2.498648,0.079762,0.657211,0.263027,Bombus_appositus
1,Bombus_lapidarius_iNat_12287097_1-jpg_jpg.rf.3...,2228.0,274.89444,1.649547,0.479759,0.702839,0.795292,-1.366018,0.370504,0.393773,...,7.749644,46.430492,25.923859,40.443417,0.069111,2.498295,0.047034,0.680557,0.272409,Bombus_lapidarius


In [9]:
test_df.head(2)

,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species
0,Bombus_robustus_GBIF_iNat_2814215056_7-jpg_jpg...,4365.0,472.55234,1.555812,0.513529,0.719703,0.766075,-0.219010,0.245637,0.357249,...,6.028357,75.442660,30.042728,42.80214,0.269145,1.205710,0.128254,0.476523,0.395222,Bombus_robustus
1,5QV06QB0AQ6K8KAK8K1KLKVK7KAKUQLSXK2K5QB0UQ305K...,2662.0,485.23254,1.596086,0.210635,0.392394,0.779395,-0.064378,0.142075,0.373467,...,7.412472,39.598804,29.143333,43.92108,0.119646,1.230382,0.061916,0.517491,0.420594,Bombus_appositus


In [10]:
train_df = fill_by_class_mean(train_df, class_col="species")
val_df   = fill_by_class_mean(val_df, class_col="species")
test_df = fill_by_class_mean(test_df, class_col="species")

X_train = train_df.drop(columns=["image", "species"])
X_val = val_df.drop(columns=["image", "species"])
X_test = test_df.drop(columns=["image", "species"])
y_train = train_df["species"]
y_val = val_df["species"]
y_test = test_df["species"]

# --- Sanity check ---
assert not X_train.isna().any().any(), "NaNs remain in X_train!"
assert not X_val.isna().any().any(), "NaNs remain in X_val!"
assert not X_test.isna().any().any(), "NaNs remain in X_val!"
print("✅ Class-wise NaN imputation complete — no missing values remain.")

✅ Class-wise NaN imputation complete — no missing values remain.


In [11]:
# ----------------- Standardize (important!) -----------------
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train.values)
X_val_np   = scaler.transform(X_val.values)
X_test_np = scaler.transform(X_test.values)

# ----------------- Class mapping (use same mapping for train & val) -----------------
classes = sorted(y_train.unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

y_train_idx = y_train.map(class_to_idx).values
y_val_idx   = y_val.map(class_to_idx).values
y_test_idx = y_test.map(class_to_idx).values

In [12]:
# ----------------- PyTorch Dataset -----------------
class ShapeFeatureDataset(Dataset):
    def __init__(self, X_np, y_np):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.long)
    def __len__(self): 
        return len(self.X)
    def __getitem__(self, idx): 
        return self.X[idx], self.y[idx]

In [13]:
train_dataset = ShapeFeatureDataset(X_train_np, y_train_idx)
val_dataset   = ShapeFeatureDataset(X_val_np,   y_val_idx)
test_dataset  = ShapeFeatureDataset(X_test_np,   y_test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

In [14]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
        # simple init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [15]:
device = torch.device(f"cuda:{DEVICE_ID}")

input_dim = X_train.shape[1]
num_classes = len(classes)

model = MLPClassifier(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [16]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [19]:
%%time

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...
 Epoch 1/100 | Train Loss: 1.3121 | Train Acc: 0.9329 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 2/100 | Train Loss: 1.3072 | Train Acc: 0.9363 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 3/100 | Train Loss: 1.3087 | Train Acc: 0.9318 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 4/100 | Train Loss: 1.3142 | Train Acc: 0.9326 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 5/100 | Train Loss: 1.3098 | Train Acc: 0.9349 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 6/100 | Train Loss: 1.3143 | Train Acc: 0.9304 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 7/100 | Train Loss: 1.3097 | Train Acc: 0.9328 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 8/100 | Train Loss: 1.3091 | Train Acc: 0.9339 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 9/100 | Train Loss: 1.3097 | Train Acc: 0.9351 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 10/100 | Train Loss: 1.3115 | Train Acc: 0.9336 | Val Loss: 4.1041 | Val Acc: 0.2142
 Epoch 11/100 | Train Loss: 1.3108 | Train Acc: 0.9326 | Val 

In [20]:
# ----------------- Load best model -----------------
# best_model_path = os.path.join(LOG_DIR, "checkpoints", "best_model.pth")
# model.load_state_dict(torch.load(best_model_path, map_location=device))
# model.to(device)
model.eval()

# ----------------- Evaluate on test set -----------------
def evaluate_test(model, loader, device):
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

test_acc = evaluate_test(model, test_loader, device)
print("Test accuracy:", test_acc)

Test accuracy: 0.2140077821011673
